In [1]:
import os
import json
import hashlib
import datetime as dt
import xml.etree.ElementTree as ET

import pandas as pd
import pyodbc
import re
from pathlib import Path

In [2]:
# Caminho base onde estão as pastas XML
XML_BASE_FOLDER = r"C:\Users\hssilva4\Horizonte Fiscal\Horizonte Fiscal - Documentos\Horizonte_Fiscal\02_-_Consultoria_Fiscal\03_-_Clientes_e_Leads\_KNX_COMERCIO_E_DISTRIBUICAO_LTDA\02_-_Coleta_Dados"
#Busca as pastas do XML
XML_FOLDERS = [
    os.path.join(XML_BASE_FOLDER, "New_extraction")

]

COMPANY_CNPJ = "46992917000110"  # só números (KNX)

SQL_CONN = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=localhost\SQLEXPRESS;"
    "DATABASE=CreditoFiscalDB;"
    "Trusted_Connection=yes;"
)

OUT_DIR = Path(r"C:\Users\hssilva4\Horizonte Fiscal\Horizonte Fiscal - Documentos\Horizonte_Fiscal\02_-_Consultoria_Fiscal\03_-_Clientes_e_Leads\_KNX_COMERCIO_E_DISTRIBUICAO_LTDA\02_-_Coleta_Dados\arquivos_processados")  # ajuste se quiser fixar
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Rastreio analítico (sem FK)
BATCH_CODE = dt.datetime.now().strftime("B%Y%m%d%H%M%S")
TENANT_CNPJ = COMPANY_CNPJ
USER_OPENID = "local-jupyter"


In [3]:
## Imports + Helpers (tipagem, datas, DocKey robusto)
DOC44_RE = re.compile(r"(?:NFe)?(\d{44})")


def only_digits(s):
    if s is None:
        return None
    d = "".join(ch for ch in str(s) if ch.isdigit())
    return d if d else None

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def safe_text(elem):
    return elem.text.strip() if elem is not None and elem.text else None

def safe_float(x):
    try:
        if x is None:
            return None
        s = str(x).strip().replace(",", ".")
        if s == "" or s.lower() == "none":
            return None
        return float(s)
    except Exception:
        return None

def parse_dt(text: str | None):
    if not text:
        return None
    t = str(text).strip()
    try:
        if t.endswith("Z"):
            t = t[:-1]
        if "+" in t:
            t = t.split("+")[0]
        if "T" in t and len(t) >= 19:
            t = t[:19]
        return dt.datetime.fromisoformat(t)
    except Exception:
        return None

def competencia_from_dt(d):
    if not d:
        return None
    return f"{d.year:04d}-{d.month:02d}"

def normalize_windows_path(p: str) -> str:
    """
    Normaliza caminho:
    - remove aspas
    - remove espaços invisíveis nas extremidades
    - troca / por \\
    - aplica prefixo \\?\ em caminhos longos no Windows
    """
    if p is None:
        return p

    p = str(p).strip().strip('"').strip("'")
    p = p.replace("/", "\\")
    p = os.path.normpath(p)

    # Windows long path
    if os.name == "nt":
        # já tem prefixo?
        if p.startswith("\\\\?\\"):
            return p
        # aplica quando estiver chegando perto do limite
        if len(p) >= 245:
            p = "\\\\?\\" + p
    return p

def path_exists(p: str) -> bool:
    try:
        return Path(normalize_windows_path(p)).exists()
    except Exception:
        return False


def to_text(x, pad: int | None = None) -> str | None:
    if x is None:
        return None
    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return None
    if pad is not None:
        d = only_digits(s)
        if d and len(d) <= pad:
            return d.zfill(pad)
    return s

def extract_44digits(text: str) -> str | None:
    m = DOC44_RE.search(text or "")
    return m.group(1) if m else None

def ensure_doc_key(doc: dict, xml_bytes: bytes, xml_path: str, content_hash: str):
    warns = []

    # 0) do parser
    k = to_text(doc.get("DocKey"), pad=44)
    if k and len(k) == 44:
        return k, warns
    
    # 1) tenta chNFe do protocolo (procNFe)
    try:
        txt = xml_bytes.decode("utf-8", errors="ignore")
        m = re.search(r"<chNFe>\s*(\d{44})\s*</chNFe>", txt)
        if m:
            warns.append("DOC_KEY_FROM_CHNFE_TAG")
            return m.group(1), warns
    except Exception:
        pass


    # 1) scan do XML inteiro (funciona p/ procNFe)
    try:
        txt = xml_bytes.decode("utf-8", errors="ignore")
        k = extract_44digits(txt)
        if k:
            warns.append("DOC_KEY_FROM_XML_TEXTSCAN")
            return k, warns
    except Exception:
        pass

    # 2) scan do nome do arquivo (tem '-'? sem problema)
    base = os.path.basename(xml_path)
    k = extract_44digits(base)
    if k:
        warns.append("DOC_KEY_FROM_FILENAME")
        return k, warns

    warns.append("DOC_KEY_FALLBACK_HASH")
    return content_hash[:44], warns

In [4]:
## BLOCO C — Listar XMLs (subpastas)
def list_xml_files_with_index(folders: list[str]):
    files = []
    index = {}  # basename_lower -> fullpath_real

    for folder in folders:
        for root, _, fns in os.walk(folder):
            for fn in fns:
                if fn.lower().endswith(".xml"):
                    full = os.path.join(root, fn)
                    full = normalize_windows_path(full)
                    files.append(full)
                    index[fn.lower()] = full  # se tiver repetido, o último vence (ok na maioria dos casos)

    return files, index
# =========================
# CONTAGEM DE XMLs (diagnóstico visual)
# =========================

def count_xml_files(folders: list[str]) -> int:
    total = 0
    for folder in folders:
        if not os.path.isdir(folder):
            continue
        for root, _, files in os.walk(folder):
            total += sum(1 for f in files if f.lower().endswith(".xml"))
    return total


total_xml = count_xml_files(XML_FOLDERS)
print(f"📦 Total de XML encontrados nas pastas: {total_xml}")

# 1) diagnóstico
total_xml = count_xml_files(XML_FOLDERS)
print(f"📦 Total de XML encontrados: {total_xml}")

# 2) listagem real para ETL
files, file_index = list_xml_files_with_index(XML_FOLDERS)
print(f"📄 XMLs preparados para leitura: {len(files)}")



📦 Total de XML encontrados nas pastas: 19371
📦 Total de XML encontrados: 19371
📄 XMLs preparados para leitura: 19371


In [5]:
## Detect + Perspectiva
def detect_doctype(root: ET.Element) -> str:
    tag = (root.tag or "").lower()

    # eventos de NF-e (cancelamento, carta de correção, etc.)
    if "proceventonfe" in tag or "envevento" in tag or "retevento" in tag:
        return "evento_nfe"

    # NF-e / procNFe
    if "nfe" in tag:
        return "nfe"

    return "unknown"


def infer_perspectiva(doc: dict) -> str:
    cnpj_emit = only_digits(doc.get("EmitenteCnpj"))
    cnpj_dest = only_digits(doc.get("DestinatarioCnpjCpf"))
    tpNF = doc.get("TpNF")

    if cnpj_emit != COMPANY_CNPJ and cnpj_dest != COMPANY_CNPJ:
        return "neutro"

    if cnpj_emit == COMPANY_CNPJ:
        base = "saida" if tpNF == 1 else "entrada"
    else:
        base = "entrada" if tpNF == 1 else "saida"

    fin = str(doc.get("FinNFe") or "").strip()
    if fin in ("3", "4"):
        return "devolucao"
    return base


In [7]:
## Parser NFe (campos completos)
def parse_nfref_list(root: ET.Element, ns: dict) -> list[str]:
    refs = []
    for n in root.findall(".//nfe:NFref/nfe:refNFe", ns):
        v = safe_text(n)
        if v:
            k = only_digits(v)
            if k and len(k) == 44:
                refs.append(k)
    # uniq
    seen, out = set(), []
    for r in refs:
        if r not in seen:
            seen.add(r); out.append(r)
    return out

def parse_evento_cancelamento(root: ET.Element):
    """
    Lê XML de evento e identifica cancelamento (tpEvento=110111).
    Retorna um header com DocKey = chNFe cancelada.
    """
    ns = {"nfe": "http://www.portalfiscal.inf.br/nfe"}

    tp = safe_text(root.find(".//nfe:tpEvento", ns))
    tp = (tp or "").strip()

    if tp != "110111":
        # não é cancelamento
        return None, [], ["EVENTO_NAO_CANCELAMENTO"]

    ch = safe_text(root.find(".//nfe:chNFe", ns))
    ch = to_text(only_digits(ch), pad=44)

    dh = safe_text(root.find(".//nfe:dhEvento", ns))
    data_cancel = parse_dt(dh)

    nprot = safe_text(root.find(".//nfe:nProt", ns))
    nprot = to_text(nprot)

    # Alguns XMLs trazem justificativa
    xjust = safe_text(root.find(".//nfe:detEvento/nfe:xJust", ns))
    xjust = to_text(xjust)

    header = {
        "DocType": "evento_cancelamento",
        "DocKey": ch,  # IMPORTANTÍSSIMO: DocKey aqui é a chave da NF-e cancelada
        "DataCancelamento": data_cancel,
        "ProtCancelamento": nprot,
        "JustCancelamento": xjust,
        "FlagCancelada": True,
    }

    return header, [], []


def parse_nfe_header_and_items(root: ET.Element):
    ns = {"nfe": "http://www.portalfiscal.inf.br/nfe"}

    inf = root.find(".//nfe:infNFe", ns)
    ide = root.find(".//nfe:ide", ns)

    doc_key = inf.attrib.get("Id") if inf is not None else None
    if doc_key and doc_key.startswith("NFe"):
        doc_key = doc_key[3:]
    doc_key = to_text(doc_key, pad=44)

    # ide
    modelo = safe_text(ide.find("nfe:mod", ns)) if ide is not None else None
    serie  = safe_text(ide.find("nfe:serie", ns)) if ide is not None else None
    numero = safe_text(ide.find("nfe:nNF", ns)) if ide is not None else None
    tpnf   = safe_text(ide.find("nfe:tpNF", ns)) if ide is not None else None
    tpnf   = int(tpnf) if tpnf and str(tpnf).isdigit() else None

    natOp    = safe_text(ide.find("nfe:natOp", ns)) if ide is not None else None
    finNFe   = safe_text(ide.find("nfe:finNFe", ns)) if ide is not None else None
    idDest   = safe_text(ide.find("nfe:idDest", ns)) if ide is not None else None
    indFinal = safe_text(ide.find("nfe:indFinal", ns)) if ide is not None else None
    indPres  = safe_text(ide.find("nfe:indPres", ns)) if ide is not None else None
    cMunFG   = safe_text(ide.find("nfe:cMunFG", ns)) if ide is not None else None

    dhEmi = safe_text(ide.find("nfe:dhEmi", ns)) if ide is not None else None
    dEmi  = safe_text(ide.find("nfe:dEmi", ns))  if ide is not None else None
    data_emissao = parse_dt(dhEmi) or parse_dt(dEmi)

    prot_dh = safe_text(root.find(".//nfe:protNFe/nfe:infProt/nfe:dhRecbto", ns))
    data_aut = parse_dt(prot_dh)

    # emit/dest
    emit_cnpj = only_digits(safe_text(root.find(".//nfe:emit/nfe:CNPJ", ns)))
    emit_nome = safe_text(root.find(".//nfe:emit/nfe:xNome", ns))
    emit_uf   = safe_text(root.find(".//nfe:emit/nfe:enderEmit/nfe:UF", ns))
    emit_crt  = safe_text(root.find(".//nfe:emit/nfe:CRT", ns))

    dest_cnpj = only_digits(safe_text(root.find(".//nfe:dest/nfe:CNPJ", ns)))
    if not dest_cnpj:
        dest_cnpj = only_digits(safe_text(root.find(".//nfe:dest/nfe:CPF", ns)))
    dest_nome = safe_text(root.find(".//nfe:dest/nfe:xNome", ns))
    dest_uf   = safe_text(root.find(".//nfe:dest/nfe:enderDest/nfe:UF", ns))

    # totais
    def tot(path): return safe_text(root.find(path, ns))
    vNF    = tot(".//nfe:total/nfe:ICMSTot/nfe:vNF")
    vProd  = tot(".//nfe:total/nfe:ICMSTot/nfe:vProd")
    vFrete = tot(".//nfe:total/nfe:ICMSTot/nfe:vFrete")
    vSeg   = tot(".//nfe:total/nfe:ICMSTot/nfe:vSeg")
    vDesc  = tot(".//nfe:total/nfe:ICMSTot/nfe:vDesc")
    vOutro = tot(".//nfe:total/nfe:ICMSTot/nfe:vOutro")

    vBC   = tot(".//nfe:total/nfe:ICMSTot/nfe:vBC")
    vICMS = tot(".//nfe:total/nfe:ICMSTot/nfe:vICMS")
    vIPI  = tot(".//nfe:total/nfe:ICMSTot/nfe:vIPI")
    vPIS  = tot(".//nfe:total/nfe:ICMSTot/nfe:vPIS")
    vCOF  = tot(".//nfe:total/nfe:ICMSTot/nfe:vCOFINS")

    # ST total
    vBCST  = tot(".//nfe:total/nfe:ICMSTot/nfe:vBCST")
    vST    = tot(".//nfe:total/nfe:ICMSTot/nfe:vST")
    vFCPST = tot(".//nfe:total/nfe:ICMSTot/nfe:vFCPST")

    base_icms  = safe_float(vBC)
    valor_icms = safe_float(vICMS)
    aliq_icms  = None
    if base_icms and valor_icms is not None and base_icms > 0:
        aliq_icms = round((valor_icms / base_icms) * 100.0, 2)

    # principal (1º item)
    det1 = root.find(".//nfe:det", ns)
    cfop_princ = safe_text(det1.find(".//nfe:prod/nfe:CFOP", ns)) if det1 is not None else None
    ncm_princ  = safe_text(det1.find(".//nfe:prod/nfe:NCM", ns))  if det1 is not None else None
    cest_princ = safe_text(det1.find(".//nfe:prod/nfe:CEST", ns)) if det1 is not None else None

    cst_princ = csosn_princ = None
    if det1 is not None:
        cst_princ   = safe_text(det1.find(".//nfe:imposto/nfe:ICMS//nfe:CST", ns))
        csosn_princ = safe_text(det1.find(".//nfe:imposto/nfe:ICMS//nfe:CSOSN", ns))

    refs = parse_nfref_list(root, ns)

    header = {
        "DocType": "nfe",
        "DocKey": doc_key,
        "Modelo": to_text(modelo),
        "Serie": to_text(serie),
        "Numero": to_text(numero),
        "TpNF": tpnf,
        "DataEmissao": data_emissao,
        "DataAutorizacao": data_aut,
        "Competencia": competencia_from_dt(data_emissao),

        "NatOp": to_text(natOp),
        "FinNFe": to_text(finNFe),
        "IdDest": to_text(idDest),
        "IndFinal": to_text(indFinal),
        "IndPres": to_text(indPres),
        "cMunFG": to_text(cMunFG),

        "EmitenteCnpj": to_text(emit_cnpj, pad=14),
        "EmitenteNome": to_text(emit_nome),
        "EmitenteUf": to_text(emit_uf),
        "EmitenteCRT": to_text(emit_crt),

        "DestinatarioCnpjCpf": to_text(dest_cnpj),
        "DestinatarioNome": to_text(dest_nome),
        "DestinatarioUf": to_text(dest_uf),

        "CfopPrincipal": to_text(cfop_princ, pad=4),
        "CstPrincipal": to_text(cst_princ or csosn_princ),
        "NcmPrincipal": to_text(ncm_princ, pad=8),
        "CestPrincipal": to_text(cest_princ),

        "ValorTotal": safe_float(vNF),
        "ValorProdutos": safe_float(vProd),
        "ValorFrete": safe_float(vFrete),
        "ValorSeguro": safe_float(vSeg),
        "ValorDesconto": safe_float(vDesc),
        "ValorOutros": safe_float(vOutro),

        "BaseIcms": base_icms,
        "ValorIcms": valor_icms,
        "AliquotaIcms": aliq_icms,
        "ValorPis": safe_float(vPIS),
        "ValorCofins": safe_float(vCOF),
        "ValorIpi": safe_float(vIPI),

        "BaseIcmsST": safe_float(vBCST),
        "ValorIcmsST": safe_float(vST),
        "ValorFcpST": safe_float(vFCPST),

        "NFeRefCount": len(refs),
        "NFeRefs": "|".join(refs) if refs else None,
    }

    items = []
    for det in root.findall(".//nfe:det", ns):
        n_item = det.attrib.get("nItem")
        prod = det.find("nfe:prod", ns)
        imp  = det.find("nfe:imposto", ns)

        def p(path): return safe_text(prod.find(path, ns)) if prod is not None else None

        cProd = p("nfe:cProd")
        cEAN  = p("nfe:cEAN")
        xProd = p("nfe:xProd")
        cfop  = p("nfe:CFOP")
        ncm   = p("nfe:NCM")
        cest  = p("nfe:CEST")
        qCom  = p("nfe:qCom")
        vUn   = p("nfe:vUnCom")
        vProd_i  = p("nfe:vProd")
        vDesc_i  = p("nfe:vDesc")
        vOutro_i = p("nfe:vOutro")

     # ICMS + ST item
        cst = csosn = orig = None
        vBC_i = vICMS_i = pICMS_i = None

        vBCST_i = vICMSST_i = pICMSST_i = pMVAST_i = None

        # >>> NOVO: ST retido anteriormente (CST 60 / CSOSN 500)
        vBCSTRet_i = vICMSSTRet_i = None
        pST_i = None
        vFCPSTRet_i = None  # opcional

        if imp is not None:
            icms_node = imp.find("nfe:ICMS", ns)
            if icms_node is not None:
                children = list(icms_node)
                grp = children[0] if children else None
                if grp is not None:
                    orig  = safe_text(grp.find(".//{*}orig"))
                    cst   = safe_text(grp.find(".//{*}CST"))
                    csosn = safe_text(grp.find(".//{*}CSOSN"))

                    vBC_i   = safe_float(safe_text(grp.find(".//{*}vBC")))
                    vICMS_i = safe_float(safe_text(grp.find(".//{*}vICMS")))
                    pICMS_i = safe_float(safe_text(grp.find(".//{*}pICMS")))

                    vBCST_i    = safe_float(safe_text(grp.find(".//{*}vBCST")))
                    vICMSST_i  = safe_float(safe_text(grp.find(".//{*}vICMSST")))
                    pICMSST_i  = safe_float(safe_text(grp.find(".//{*}pICMSST")))
                    pMVAST_i   = safe_float(safe_text(grp.find(".//{*}pMVAST")))

                    # >>> NOVO: campos de ST retido anteriormente
                    vBCSTRet_i   = safe_float(safe_text(grp.find(".//{*}vBCSTRet")))
                    vICMSSTRet_i = safe_float(safe_text(grp.find(".//{*}vICMSSTRet")))
                    pST_i        = safe_float(safe_text(grp.find(".//{*}pST")))  # quando existir
                    vFCPSTRet_i  = safe_float(safe_text(grp.find(".//{*}vFCPSTRet")))  # opcional


        vIPI_i = safe_float(safe_text(imp.find(".//nfe:IPI//nfe:vIPI", ns)) if imp is not None else None)
        vPIS_i = safe_float(safe_text(imp.find(".//nfe:PIS//nfe:vPIS", ns)) if imp is not None else None)
        vCOF_i = safe_float(safe_text(imp.find(".//nfe:COFINS//nfe:vCOFINS", ns)) if imp is not None else None)
        # ST ajustado: usa ST da operação, senão usa ST retido anteriormente
        base_st_aj = vBCST_i if (vBCST_i and vBCST_i > 0) else vBCSTRet_i
        valor_st_aj = vICMSST_i if (vICMSST_i and vICMSST_i > 0) else vICMSSTRet_i

        flag_st = True if (
            (valor_st_aj and valor_st_aj > 0) or (base_st_aj and base_st_aj > 0)
        ) else False


        items.append({
            "DocKey": doc_key,
            "ItemNumber": int(n_item) if n_item and str(n_item).isdigit() else None,
            "cProd": to_text(cProd),
            "cEAN": to_text(cEAN),
            "Descricao": to_text(xProd),
            "Cfop": to_text(cfop, pad=4),
            "Ncm": to_text(ncm, pad=8),
            "Cest": to_text(cest),
            "Cst": to_text(cst),
            "Csosn": to_text(csosn),
            "Orig": to_text(orig),

            "qCom": safe_float(qCom),
            "vUnCom": safe_float(vUn),
            "ValorItem": safe_float(vProd_i),
            "vDescItem": safe_float(vDesc_i),
            "vOutroItem": safe_float(vOutro_i),

            "BaseIcms": vBC_i,
            "ValorIcms": vICMS_i,
            "AliquotaIcms": pICMS_i,

            "BaseIcmsST": vBCST_i,
            "ValorIcmsST": vICMSST_i,
            "AliquotaIcmsST": pICMSST_i,
            "MvaST": pMVAST_i,
            # >>> NOVO
            "BaseIcmsSTRet": vBCSTRet_i,
            "ValorIcmsSTRet": vICMSSTRet_i,
            "AliquotaST": pST_i,
            "ValorFcpSTRet": vFCPSTRet_i,  # opcional

            # >>> NOVO (facilita muito no Power BI)
            "BaseIcmsST_Ajustada": base_st_aj,
            "ValorIcmsST_Ajustada": valor_st_aj,

            "FlagItemComST": flag_st,


            "ValorIpi": vIPI_i,
            "ValorPis": vPIS_i,
            "ValorCofins": vCOF_i,
          
        })

    total_bcst_ret = sum((i.get("BaseIcmsSTRet") or 0) for i in items)
    total_st_ret   = sum((i.get("ValorIcmsSTRet") or 0) for i in items)

    header["BaseIcmsSTRet"]  = total_bcst_ret
    header["ValorIcmsSTRet"] = total_st_ret
    header["ValorIcmsST_TotalAjustado"] = (header.get("ValorIcmsST") or 0) + total_st_ret

    return header, items


In [8]:
##parse_any + build dataframes + validações

def parse_any_bytes(xml_bytes: bytes):
    try:
        root = ET.fromstring(xml_bytes)
    except Exception as e:
        return {"DocType":"unknown","DocKey":None}, [], [f"XML_PARSE_ERROR: {e}"]

    doctype = detect_doctype(root)

    try:
        if doctype == "nfe":
            h, items = parse_nfe_header_and_items(root)
            return h, items, []

        if doctype == "evento_nfe":
            h, items, warns = parse_evento_cancelamento(root)
            if h is None:
                return {"DocType": "evento_nfe", "DocKey": None}, [], warns
            return h, items, warns

        return {"DocType": doctype, "DocKey": None}, [], [f"UNSUPPORTED_DOCTYPE: {doctype}"]

    except Exception as e:
        return {"DocType": doctype, "DocKey": None}, [], [f"PARSER_ERROR: {e}"]


def parse_any(xml_path: str):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception as e:
        return {"DocType":"unknown","DocKey":None}, [], [f"XML_PARSE_ERROR: {e}"]

    doctype = detect_doctype(root)

    try:
        if doctype == "nfe":
            h, items = parse_nfe_header_and_items(root)
            return h, items, []

        if doctype == "evento_nfe":
            h, items, warns = parse_evento_cancelamento(root)
            if h is None:
                return {"DocType": "evento_nfe", "DocKey": None}, [], warns
            return h, items, warns

        return {"DocType": doctype, "DocKey": None}, [], [f"UNSUPPORTED_DOCTYPE: {doctype}"]

    except Exception as e:
        return {"DocType": doctype, "DocKey": None}, [], [f"PARSER_ERROR: {e}"]

    
## Leitura robusta para evitar Read_error    
def read_xml_bytes_safe(path: str, file_index: dict[str, str] | None = None):
    """
    Tenta ler o arquivo:
    1) pelo path original (normalizado)
    2) pelo \\?\ (se necessário)
    3) fallback: tenta localizar pelo basename via index
    Retorna (bytes, None) ou (None, mensagem_erro)
    """
    try_paths = []

    if path:
        p = normalize_windows_path(path)
        try_paths.append(p)

        # se não tiver prefixo e for windows, tenta versão com \\?\ também
        if os.name == "nt" and not p.startswith("\\\\?\\"):
            try_paths.append("\\\\?\\" + os.path.normpath(str(p)))

    # tenta abrir nos paths candidatos
    for p in try_paths:
        try:
            with open(p, "rb") as f:
                return f.read(), None
        except FileNotFoundError:
            continue
        except OSError as e:
            # pode ser path longo/permission/etc
            last_err = f"OS_ERROR: {e}"
            continue
        except Exception as e:
            return None, f"READ_ERROR: {e}"

    # fallback: acha pelo basename
    if file_index is not None and path:
        base = os.path.basename(str(path)).lower()
        real = file_index.get(base)
        if real:
            try:
                with open(real, "rb") as f:
                    return f.read(), None
            except Exception as e:
                return None, f"READ_ERROR_FALLBACK_BASENAME: {e}"

    return None, f"READ_ERROR: File not found after normalization. path={path}"


def build_dataframes(files: list[str], file_index: dict[str, str] | None = None, origem="XML"):
    docs_rows, items_rows, val_rows = [], [], []

    for path in files:
        content, err = read_xml_bytes_safe(path, file_index=file_index)

        if err:
            docs_rows.append({
                "FonteXML": origem,
                "DocKey": None,
                "DocType": "unknown",
                "Status": "invalid",
                "StorageKey": normalize_windows_path(path),
                "OriginalFilename": os.path.basename(str(path)),
                "Errors": json.dumps([err], ensure_ascii=False),
                "Warnings": None
            })
            continue

        content_hash = sha256_bytes(content)

        doc, items, warns = parse_any_bytes(content)

        warns = warns or []

        doc_key, extra = ensure_doc_key(doc, content, path, content_hash)
        doc["DocKey"] = doc_key
        warns.extend(extra)

        doc["ClientePerspectiva"] = infer_perspectiva(doc)
        status = "valid" if doc.get("DocType") == "nfe" else "partial"

        docs_rows.append({
            "FonteXML": origem,
            **doc,
            "ClientePerspectiva": doc.get("ClientePerspectiva"),
            "Status": status,
            "ContentHash": content_hash,
            "StorageKey": normalize_windows_path(path),
            "OriginalFilename": os.path.basename(str(path)),
            "Errors": json.dumps(
                [w for w in warns if "ERROR" in w or "PARSER" in w or "XML_PARSE" in w],
                ensure_ascii=False
            ) if warns else None,
            "Warnings": json.dumps(
                [w for w in warns if not ("ERROR" in w or "PARSER" in w or "XML_PARSE" in w)],
                ensure_ascii=False
            ) if warns else None,
        })

        for it in items or []:
            items_rows.append({**it})

        if not doc_key or len(str(doc_key)) != 44:
            val_rows.append({
                "DocKey": doc_key,
                "Regra": "DOC_001",
                "Severidade": "error",
                "Titulo": "DocKey inválido",
                "Descricao": f"DocKey={doc_key}",
                "Arquivo": os.path.basename(str(path))
            })

    return (
        pd.DataFrame(docs_rows),
        pd.DataFrame(items_rows),
        pd.DataFrame(val_rows)
    )



In [9]:
## doc_referencias em linhas (DocKey → DocKeyRef)
def build_doc_referencias(df_docs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if "NFeRefs" not in df_docs.columns:
        return pd.DataFrame(columns=["DocKey","DocKeyRef","FinNFe","TpNF","Competencia","FlagDevolucao"])

    for _, r in df_docs.iterrows():
        refs = r.get("NFeRefs")
        if not refs or pd.isna(refs):
            continue
        for ref in str(refs).split("|"):
            k = only_digits(ref)
            if k and len(k) == 44:
                rows.append({
                    "DocKey": r.get("DocKey"),
                    "DocKeyRef": k,
                    "FinNFe": r.get("FinNFe"),
                    "TpNF": r.get("TpNF"),
                    "Competencia": r.get("Competencia"),
                    "FlagDevolucao": True if str(r.get("FinNFe") or "").strip() in ("3","4") else False
                })

    df = pd.DataFrame(rows)
    if df.empty:
        df = pd.DataFrame(columns=["DocKey","DocKeyRef","FinNFe","TpNF","Competencia","FlagDevolucao"])

    df["DocKey"] = df["DocKey"].astype("string")
    df["DocKeyRef"] = df["DocKeyRef"].astype("string")
    df["FinNFe"] = df["FinNFe"].astype("string")
    df["Competencia"] = df["Competencia"].astype("string")
    df["TpNF"] = pd.to_numeric(df["TpNF"], errors="coerce").astype("Int64")
    df["FlagDevolucao"] = df["FlagDevolucao"].fillna(False).astype("boolean")
    return df

def apply_cancelamentos(df_docs: pd.DataFrame) -> pd.DataFrame:
    if df_docs.empty or "DocType" not in df_docs.columns or "DocKey" not in df_docs.columns:
        return df_docs

    # conjunto de chaves canceladas a partir dos eventos
    ev = df_docs[df_docs["DocType"].astype("string") == "evento_cancelamento"].copy()
    if ev.empty:
        # não há eventos de cancelamento nos XMLs lidos
        df_docs["FlagCancelada"] = df_docs.get("FlagCancelada", False)
        return df_docs

    ev["DocKey"] = ev["DocKey"].astype("string")
    canceladas = set(ev["DocKey"].dropna().tolist())

    # mapa de informações do evento (pega o "primeiro" por chave)
    ev_min = (
        ev.sort_values(by=["DataCancelamento"], na_position="last")
          .drop_duplicates(subset=["DocKey"], keep="first")
          .set_index("DocKey")
    )

    df_docs["DocKey"] = df_docs["DocKey"].astype("string")
    mask_nfe_cancelada = (df_docs["DocType"].astype("string") == "nfe") & (df_docs["DocKey"].isin(canceladas))

    # Flag e colunas de evidência
    df_docs["FlagCancelada"] = df_docs.get("FlagCancelada", False)
    df_docs.loc[mask_nfe_cancelada, "FlagCancelada"] = True

    # traz data/protocolo do evento para a linha da NF-e
    df_docs.loc[mask_nfe_cancelada, "DataCancelamento"] = df_docs.loc[mask_nfe_cancelada, "DocKey"].map(
        lambda k: ev_min.loc[k, "DataCancelamento"] if k in ev_min.index else pd.NaT
    )
    df_docs.loc[mask_nfe_cancelada, "ProtCancelamento"] = df_docs.loc[mask_nfe_cancelada, "DocKey"].map(
        lambda k: ev_min.loc[k, "ProtCancelamento"] if k in ev_min.index else None
    )

    # >>> AQUI: evidenciar na ClientePerspectiva <<<
    # exemplo: "saida" -> "saida_cancelada", "entrada" -> "entrada_cancelada", "devolucao" -> "devolucao_cancelada"
    cp = df_docs["ClientePerspectiva"].astype("string")
    df_docs.loc[mask_nfe_cancelada, "ClientePerspectiva"] = (cp[mask_nfe_cancelada].fillna("desconhecida") + "_cancelada")

    return df_docs

df_docs_raw, df_items_raw, df_validacoes_raw = build_dataframes(
    files,
    file_index=file_index,
    origem="XML_out_25_extraidos"
)

# >>> APLICA CANCELAMENTOS (marca ClientePerspectiva) <<<
df_docs_raw = apply_cancelamentos(df_docs_raw)

df_doc_refs = build_doc_referencias(df_docs_raw)



In [10]:
##Tipagem final + salvar em arquivos_processados (Power BI consome)
def enforce_schema(df_docs, df_items, df_validacoes):
    doc_string = [
        "FonteXML","DocType","DocKey","Modelo","Serie","Numero","Competencia",
        "NatOp","FinNFe","IdDest","IndFinal","IndPres","cMunFG",
        "EmitenteCnpj","EmitenteNome","EmitenteUf","EmitenteCRT",
        "DestinatarioCnpjCpf","DestinatarioNome","DestinatarioUf",
        "CfopPrincipal","CstPrincipal","NcmPrincipal","CestPrincipal",
        "NFeRefs","ClientePerspectiva",
        "ProtCancelamento","JustCancelamento",
        "Status","ContentHash","StorageKey","OriginalFilename",
        "Errors","Warnings"
    ]

    doc_dt = ["DataEmissao","DataAutorizacao","DataCancelamento"]

    # ... (seu código atual)

    doc_bool = ["FlagCancelada"]
    for c in doc_bool:
        if c in df_docs.columns:
            df_docs[c] = df_docs[c].fillna(False).astype("boolean")

    return df_docs, df_items, df_validacoes

    doc_float = [
        "ValorTotal","ValorProdutos","ValorFrete","ValorSeguro","ValorDesconto","ValorOutros",
        "BaseIcms","ValorIcms","AliquotaIcms","ValorPis","ValorCofins","ValorIpi",
        "BaseIcmsST","ValorIcmsST","ValorFcpST"
    ]
    doc_int = ["TpNF","NFeRefCount"]
    doc_dt = ["DataEmissao","DataAutorizacao"]

    for c in doc_string:
        if c in df_docs.columns: df_docs[c] = df_docs[c].astype("string")
    for c in doc_float:
        if c in df_docs.columns: df_docs[c] = pd.to_numeric(df_docs[c], errors="coerce")
    for c in doc_int:
        if c in df_docs.columns: df_docs[c] = pd.to_numeric(df_docs[c], errors="coerce").astype("Int64")
    for c in doc_dt:
        if c in df_docs.columns: df_docs[c] = pd.to_datetime(df_docs[c], errors="coerce")

    # items
    item_string = ["DocKey","cProd","cEAN","Descricao","Cfop","Ncm","Cest","Cst","Csosn","Orig"]
    item_float = [
        "qCom","vUnCom","ValorItem","vDescItem","vOutroItem",
        "BaseIcms","ValorIcms","AliquotaIcms",
        "BaseIcmsST","ValorIcmsST","AliquotaIcmsST","MvaST",
        "ValorIpi","ValorPis","ValorCofins"
    ]
    item_int = ["ItemNumber"]

    for c in item_string:
        if c in df_items.columns: df_items[c] = df_items[c].astype("string")
    for c in item_float:
        if c in df_items.columns: df_items[c] = pd.to_numeric(df_items[c], errors="coerce")
    for c in item_int:
        if c in df_items.columns: df_items[c] = pd.to_numeric(df_items[c], errors="coerce").astype("Int64")
    if "FlagItemComST" in df_items.columns:
        df_items["FlagItemComST"] = df_items["FlagItemComST"].fillna(False).astype("boolean")

    # validações
    val_string = ["DocKey","Regra","Severidade","Titulo","Descricao","Arquivo"]
    for c in val_string:
        if c in df_validacoes.columns: df_validacoes[c] = df_validacoes[c].astype("string")

    return df_docs, df_items, df_validacoes


# -------- EXECUÇÃO + SALVAR ----------
run_id = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

files, file_index = list_xml_files_with_index(XML_FOLDERS)
print("XMLs encontrados:", len(files))


df_docs_raw, df_items_raw, df_validacoes_raw = build_dataframes(
    files,
    file_index=file_index,          # <- novo
    origem="XML_out_25_extraidos"
)

df_doc_refs = build_doc_referencias(df_docs_raw)


# tipar
df_docs_clean, df_items_clean, df_validacoes_clean = enforce_schema(df_docs_raw, df_items_raw, df_validacoes_raw)

# =========================
# EXPORT POWER BI SAFE (CSV)
# =========================
from pathlib import Path

OUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV para Power BI (NÃO usar excel_lock aqui)
df_docs_clean.to_csv(
    OUT_DIR / f"{run_id}_docs_powerbi.csv",
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    lineterminator="\n"
)

df_items_clean.to_csv(
    OUT_DIR / f"{run_id}_items_powerbi.csv",
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    lineterminator="\n"
)

df_doc_refs.to_csv(
    OUT_DIR / f"{run_id}_doc_referencias_powerbi.csv",
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    lineterminator="\n"
)

df_validacoes_clean.to_csv(
    OUT_DIR / f"{run_id}_validacoes_powerbi.csv",
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    lineterminator="\n"
)

print("✅ CSVs PowerBI-safe gerados em:", OUT_DIR)

# # salvar (Power BI safe)
# df_docs_clean.to_parquet(OUT_DIR / f"{run_id}_docs.parquet", index=False)
# df_items_clean.to_parquet(OUT_DIR / f"{run_id}_items.parquet", index=False)
# df_doc_refs.to_parquet(OUT_DIR / f"{run_id}_doc_referencias.parquet", index=False)
# df_validacoes_clean.to_parquet(OUT_DIR / f"{run_id}_validacoes.parquet", index=False)

# print("✅ Arquivos salvos em:", OUT_DIR)
# print("Docs:", len(df_docs_clean), "Itens:", len(df_items_clean), "Refs:", len(df_doc_refs), "Validações:", len(df_validacoes_clean))

# # diagnóstico DocKey (pra você bater o olho)
# bad = df_docs_clean[df_docs_clean["DocKey"].isna() | (df_docs_clean["DocKey"].astype("string").str.len() != 44)]
# print("DocKey ruins:", len(bad))
# if len(bad) > 0:
#     print(bad[["OriginalFilename","StorageKey","DocType","Errors","Warnings"]].head(30).to_string(index=False))


XMLs encontrados: 19371
✅ CSVs PowerBI-safe gerados em: C:\Users\hssilva4\Horizonte Fiscal\Horizonte Fiscal - Documentos\Horizonte_Fiscal\02_-_Consultoria_Fiscal\03_-_Clientes_e_Leads\_KNX_COMERCIO_E_DISTRIBUICAO_LTDA\02_-_Coleta_Dados\arquivos_processados
